# DISQO Jobs Scraper

**Source:** [disqo.com/careers](https://www.disqo.com/careers/)  
**Backend ATS:** Lever (public API, no authentication required)  
**API:** `https://api.lever.co/v0/postings/disqo?mode=json`  
**Filter:** Yerevan / Armenia positions only

### Why DISQO separately?
DISQO is a US-headquartered ad measurement company with an engineering office in Yerevan, Armenia. Their jobs are posted on their own Lever-powered careers page rather than on local job boards.

### API structure
- Lever's public job board API returns all live postings in a single request (no pagination)
- Each job object contains: `text` (title), `categories` (location, team, commitment), `descriptionPlain`, `openingPlain`, `additionalPlain`, `lists` (structured sections), `hostedUrl`
- Full text is assembled from `descriptionPlain` + `openingPlain` + `additionalPlain` + `lists` content

### Ethics & robots.txt
- Lever public board API is open by design — publicly documented, no authentication needed
- `robots.txt` at disqo.com does not restrict `/careers/` paths (checked 2026-03-22)
- Single API call, no HTML crawling

### Output files
- `data/raw/jobs/disqo_jobs_raw.csv`
- `data/processed/jobs/disqo_jobs_standardized.csv`

In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

LEVER_URL = "https://api.lever.co/v0/postings/disqo"
HEADERS   = {"User-Agent": "ThesisResearch/1.0 (Armenian IT curriculum alignment; academic use)"}

BASE_DIR = Path.cwd()
RAW_DIR  = BASE_DIR / "data" / "raw" / "jobs"
PROC_DIR = BASE_DIR / "data" / "processed" / "jobs"

print(f"Lever API: {LEVER_URL}?mode=json")
print(f"Raw output:  {RAW_DIR}")
print(f"Proc output: {PROC_DIR}")

Lever API: https://api.lever.co/v0/postings/disqo?mode=json
Raw output:  /Users/lianaaghamalyan/thesis_data/data/raw/jobs
Proc output: /Users/lianaaghamalyan/thesis_data/data/processed/jobs


## Helper functions

In [2]:
def html_to_text(html_str):
    if not html_str: return ""
    text = BeautifulSoup(str(html_str), "html.parser").get_text("\n", strip=True)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def build_full_text(j):
    """Assemble full NLP text from all Lever job content fields."""
    parts = []
    for field in ["descriptionPlain", "openingPlain", "additionalPlain"]:
        val = (j.get(field) or "").strip()
        if val:
            parts.append(val)
    for lst in (j.get("lists") or []):
        heading = lst.get("text", "")
        content = html_to_text(lst.get("content", ""))
        if content:
            parts.append(f"{heading}:\n{content}")
    return "\n\n".join(parts).strip()

def is_armenia(j):
    loc = j.get("categories", {}).get("location", "").lower()
    return "yerevan" in loc or "armenia" in loc

print("Helpers defined.")

Helpers defined.


## Step 1 — Fetch all jobs from Lever API

In [3]:
resp = requests.get(LEVER_URL, headers=HEADERS, params={"mode": "json"}, timeout=15)
resp.raise_for_status()
all_jobs = resp.json()

print(f"Total Disqo jobs: {len(all_jobs)}")
print()
for j in all_jobs:
    loc = j.get("categories", {}).get("location", "N/A")
    print(f"  {j['text']:65s} | {loc}")

Total Disqo jobs: 11

  Associate General Counsel                                         | Los Angeles, CA
  Business Development Representative (BDR)                         | Los Angeles, CA
  Director of Sales, Brand Direct & Agency Partnerships             | New York, NY
  GTM Engineer                                                      | Los Angeles, CA
  Head of Marketing                                                 | Los Angeles, CA
  Insights Director - Ad Measurement                                | New York, NY
  Insights Manager - Ad Measurement                                 | New York, NY
  Member Support Specialist (Evening Shift)                         | Yerevan
  Sr. Insights Manager - Ad Measurement                             | Los Angeles, CA
  Sr. Staff Software Engineer - Advanced Analytics Platform         | Los Angeles, CA
  TPM Director - Ad Measurement                                     | Los Angeles, CA


## Step 2 — Filter Armenia and build records

In [4]:
armenia = [j for j in all_jobs if is_armenia(j)]
print(f"Armenia jobs: {len(armenia)}")
print()

records = []
for j in armenia:
    cats      = j.get("categories", {})
    full_text = build_full_text(j)
    print(f"  {j['text']}")
    print(f"  Team: {cats.get('team','')} | Commitment: {cats.get('commitment','')}")
    print(f"  full_text: {len(full_text)} chars")
    records.append({
        "source":          "disqo",
        "source_url":      j.get("hostedUrl", ""),
        "job_title":       j.get("text", ""),
        "company_name":    "DISQO",
        "location":        cats.get("location", ""),
        "department":      cats.get("team", ""),
        "employment_type": cats.get("commitment", ""),
        "posting_date":    "",
        "full_text":       full_text,
    })

Armenia jobs: 1

  Member Support Specialist (Evening Shift)
  Team: Customer Support | Commitment: Full Time
  full_text: 6721 chars


## Step 3 — Save outputs

In [5]:
df = pd.DataFrame(records)
raw_path = RAW_DIR / "disqo_jobs_raw.csv"
df.to_csv(raw_path, index=False, encoding="utf-8")
print(f"Raw saved: {raw_path} ({len(df)} rows)")

std_cols = ["source","source_url","job_title","company_name","location",
            "department","employment_type","posting_date","full_text"]
std_df = df[std_cols]
std_path = PROC_DIR / "disqo_jobs_standardized.csv"
std_df.to_csv(std_path, index=False, encoding="utf-8")
print(f"Standardized saved: {std_path} ({len(std_df)} rows)")
print()
print("=== FIELD COVERAGE ===")
for col in std_cols:
    filled = std_df[col].replace("", pd.NA).notna().sum()
    print(f"  {col:20s}: {filled}/{len(std_df)} ({filled/len(std_df)*100:.0f}%)")
print()
ft = std_df["full_text"].str.len()
print(f"full_text — {ft.iloc[0]} chars")

Raw saved: /Users/lianaaghamalyan/thesis_data/data/raw/jobs/disqo_jobs_raw.csv (1 rows)
Standardized saved: /Users/lianaaghamalyan/thesis_data/data/processed/jobs/disqo_jobs_standardized.csv (1 rows)

=== FIELD COVERAGE ===
  source              : 1/1 (100%)
  source_url          : 1/1 (100%)
  job_title           : 1/1 (100%)
  company_name        : 1/1 (100%)
  location            : 1/1 (100%)
  department          : 1/1 (100%)
  employment_type     : 1/1 (100%)
  posting_date        : 0/1 (0%)
  full_text           : 1/1 (100%)

full_text — 6721 chars
